# EDA датасета ESC-50

Разведочный анализ данных ESC-50 для классификации звуков окружающей среды.

Ожидаемая локальная структура:

```text
data/raw/ESC-50/
├── esc50.csv
└── audio/audio/
    ├── *.wav
    ├── 44100/*.wav
    └── 16000/*.wav
```

Ноутбук также поддерживает типичную структуру Kaggle с `esc50.csv` и `audio/audio/*.wav`.

## Настройка

In [ ]:
from pathlib import Path

import IPython.display as ipd
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import soundfile as sf
from tqdm.auto import tqdm

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_rows', 80)
pd.set_option('display.max_columns', 40)
RANDOM_STATE = 42

In [ ]:
def find_first_existing(candidates):
    return next((path for path in candidates if path.exists()), None)

repo_root = Path.cwd()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent

metadata_candidates = [
    Path('/kaggle/input/environmental-sound-classification-50/esc50.csv'),
    Path('/kaggle/input/esc50/esc50.csv'),
    repo_root / 'data' / 'raw' / 'ESC-50' / 'esc50.csv',
    repo_root / 'data' / 'raw' / 'ESC50' / 'esc50.csv',
    repo_root / 'esc50.csv',
]

audio_root_candidates = [
    Path('/kaggle/input/environmental-sound-classification-50/audio/audio'),
    Path('/kaggle/input/esc50/audio/audio'),
    repo_root / 'data' / 'raw' / 'ESC-50' / 'audio' / 'audio',
    repo_root / 'data' / 'raw' / 'ESC50' / 'audio' / 'audio',
    repo_root / 'audio' / 'audio',
]

metadata_path = find_first_existing(metadata_candidates)
audio_root = find_first_existing(audio_root_candidates)

if metadata_path is None:
    raise FileNotFoundError('esc50.csv was not found. Check dataset location or update metadata_candidates.')
if audio_root is None:
    raise FileNotFoundError('ESC-50 audio directory was not found. Check dataset location or update audio_root_candidates.')

preferred_audio_dirs = [audio_root / '44100', audio_root, audio_root / '16000']
audio_dir = find_first_existing([path for path in preferred_audio_dirs if path.is_dir()])

print(f'Корень репозитория: {repo_root}')
print(f'Путь к метаданным: {metadata_path}')
print(f'Корневая директория аудио: {audio_root}')
print(f'Выбранная директория аудио: {audio_dir}')
print(f'WAV-файлов в выбранной директории: {len(list(audio_dir.glob("*.wav")))}')

## Загрузка метаданных

In [ ]:
esc = pd.read_csv(metadata_path).copy()

expected_columns = {'filename', 'fold', 'target', 'category', 'esc10', 'src_file', 'take'}
missing_columns = expected_columns.difference(esc.columns)
if missing_columns:
    raise ValueError(f'В esc50.csv отсутствуют колонки: {sorted(missing_columns)}')

esc['audio_path'] = esc['filename'].map(lambda filename: audio_dir / filename)
missing_audio = esc.loc[~esc['audio_path'].map(Path.exists), ['filename', 'audio_path']]
if not missing_audio.empty:
    display(missing_audio.head())
    raise FileNotFoundError(f'Не найдено {len(missing_audio)} аудиофайлов в {audio_dir}')

coarse_group_by_target = {
    range(0, 10): 'animals',
    range(10, 20): 'natural_soundscapes_water',
    range(20, 30): 'human_non_speech',
    range(30, 40): 'interior_domestic',
    range(40, 50): 'exterior_urban',
}

def target_to_group(target):
    for target_range, group in coarse_group_by_target.items():
        if target in target_range:
            return group
    return 'unknown'

esc['coarse_group'] = esc['target'].map(target_to_group)
esc['esc10'] = esc['esc10'].astype(bool)

print(f'Строк: {len(esc):,}')
print(f'Классов: {esc["category"].nunique()}')
print(f'Folds: {sorted(esc["fold"].unique())}')
print(f'Строк в подмножестве ESC-10: {esc["esc10"].sum():,}')
display(esc.head())

## Классы

In [ ]:
classes = (
    esc[['target', 'category', 'coarse_group', 'esc10']]
    .drop_duplicates()
    .sort_values('target')
    .reset_index(drop=True)
)

class_counts = (
    esc.groupby(['target', 'category', 'coarse_group', 'esc10'])
    .size()
    .reset_index(name='num_samples')
    .sort_values('target')
    .reset_index(drop=True)
)

display(class_counts)
print('Названия классов:')
print(', '.join(classes['category'].tolist()))

ESC-50 содержит 50 классов по 40 клипов в каждом. Целевые метки сгруппированы в пять укрупнённых блоков по id класса: животные, природные звуки и вода, неречевая активность человека, бытовые звуки в помещении, внешние и городские звуки.

## Баланс датасета

In [ ]:
fold_counts = esc['fold'].value_counts().sort_index().rename_axis('fold').reset_index(name='num_samples')
coarse_counts = esc['coarse_group'].value_counts().rename_axis('coarse_group').reset_index(name='num_samples')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.barplot(data=class_counts, x='num_samples', y='category', hue='coarse_group', dodge=False, ax=axes[0])
axes[0].set_title('Количество клипов по классам')
axes[0].set_xlabel('Клипы')
axes[0].set_ylabel('Класс')
axes[0].legend(title='Группа', bbox_to_anchor=(1.02, 1), loc='upper left')

sns.barplot(data=fold_counts, x='fold', y='num_samples', color='#4C78A8', ax=axes[1])
axes[1].set_title('Количество клипов по fold')
axes[1].set_xlabel('Fold')
axes[1].set_ylabel('Клипы')

plt.tight_layout()
plt.show()

display(fold_counts)
display(coarse_counts)

Локальные метаданные идеально сбалансированы: в каждом классе по 40 клипов, а в каждом fold по 400 клипов. Это удобно для кросс-валидации и сравнения моделей, потому что базовое распределение классов не смещает метрику.

## Структура fold-разбиения

In [ ]:
fold_category = esc.groupby(['fold', 'category']).size().unstack(fill_value=0)
fold_group = esc.groupby(['fold', 'coarse_group']).size().reset_index(name='num_samples')
esc10_fold = esc.groupby(['fold', 'esc10']).size().reset_index(name='num_samples')

fig, axes = plt.subplots(1, 2, figsize=(17, 5))
sns.heatmap(fold_category, cmap='mako', linewidths=0.2, cbar_kws={'label': 'клипы'}, ax=axes[0])
axes[0].set_title('Распределение классов по fold')
axes[0].set_xlabel('Класс')
axes[0].set_ylabel('Fold')

sns.barplot(data=fold_group, x='fold', y='num_samples', hue='coarse_group', ax=axes[1])
axes[1].set_title('Укрупнённые группы по fold')
axes[1].set_xlabel('Fold')
axes[1].set_ylabel('Клипы')
axes[1].legend(title='Группа', bbox_to_anchor=(1.02, 1), loc='upper left')

plt.tight_layout()
plt.show()

display(esc10_fold)

Каждый fold равномерно покрывает все классы. Подмножество ESC-10 также равномерно распределено по fold, поэтому официальное разбиение подходит для экспериментов и с ESC-50, и с ESC-10.

## Разнообразие источников

In [ ]:
source_counts = esc['src_file'].value_counts().head(20).rename_axis('src_file').reset_index(name='num_clips')
unique_sources = (
    esc.groupby(['category', 'coarse_group'])['src_file']
    .nunique()
    .reset_index(name='num_unique_sources')
    .sort_values('num_unique_sources', ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(17, 6))
sns.barplot(data=source_counts, x='num_clips', y='src_file', color='#F58518', ax=axes[0])
axes[0].set_title('Топ исходных файлов по числу клипов')
axes[0].set_xlabel('Клипы')
axes[0].set_ylabel('Исходный файл')

sns.barplot(data=unique_sources, x='num_unique_sources', y='category', hue='coarse_group', dodge=False, ax=axes[1])
axes[1].set_title('Уникальные исходные файлы по классам')
axes[1].set_xlabel('Уникальные исходные файлы')
axes[1].set_ylabel('Класс')
axes[1].legend(title='Группа', bbox_to_anchor=(1.02, 1), loc='upper left')

plt.tight_layout()
plt.show()

display(unique_sources.head(10))

Баланс классов не означает одинаковое акустическое разнообразие. Некоторые классы собраны из большего числа уникальных исходных файлов, поэтому variation на уровне источников стоит учитывать при интерпретации результатов валидации.

## Аудио-метаданные

In [ ]:
audio_rows = []
for row in tqdm(esc.itertuples(index=False), total=len(esc), desc='Чтение аудио-заголовков'):
    info = sf.info(row.audio_path)
    audio_rows.append(
        {
            'filename': row.filename,
            'target': row.target,
            'category': row.category,
            'coarse_group': row.coarse_group,
            'fold': row.fold,
            'duration_seconds': info.duration,
            'sample_rate': info.samplerate,
            'channels': info.channels,
            'frames': info.frames,
            'format': info.format,
            'subtype': info.subtype,
        }
    )

esc_audio = pd.DataFrame(audio_rows)
display(esc_audio[['duration_seconds', 'sample_rate', 'channels', 'frames']].describe().T)
display(esc_audio[['sample_rate', 'channels', 'format', 'subtype']].value_counts().reset_index(name='num_samples'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.histplot(data=esc_audio, x='duration_seconds', bins=20, color='#54A24B', ax=axes[0])
axes[0].set_title('Длительность клипов')
axes[0].set_xlabel('Секунды')
axes[0].set_ylabel('Клипы')

sr_counts = esc_audio['sample_rate'].value_counts().sort_index().rename_axis('sample_rate').reset_index(name='num_samples')
sns.barplot(data=sr_counts, x='sample_rate', y='num_samples', color='#E45756', ax=axes[1])
axes[1].set_title('Частоты дискретизации')
axes[1].set_xlabel('Частота дискретизации')
axes[1].set_ylabel('Клипы')

plt.tight_layout()
plt.show()

Клипы ESC-50 технически стандартизированы, что упрощает preprocessing. В реальных потоках длительность клипа, громкость, уровень шума и частота дискретизации обычно менее контролируемы, поэтому этот benchmark не стоит считать полноценной production-заменой.

## Волновые формы и mel-спектрограммы

In [ ]:
example_rows = (
    esc.sort_values(['coarse_group', 'target', 'filename'])
    .groupby('coarse_group', as_index=False)
    .first()
    .sort_values('target')
)

fig, axes = plt.subplots(len(example_rows), 2, figsize=(15, 3.2 * len(example_rows)))
if len(example_rows) == 1:
    axes = np.array([axes])

for row_idx, row in enumerate(example_rows.itertuples(index=False)):
    signal, sr = librosa.load(row.audio_path, sr=None, mono=True)
    librosa.display.waveshow(signal, sr=sr, ax=axes[row_idx, 0])
    axes[row_idx, 0].set_title(f'Волновая форма: {row.coarse_group} / {row.category}')
    axes[row_idx, 0].set_xlabel('Время')
    axes[row_idx, 0].set_ylabel('Амплитуда')

    mel = librosa.feature.melspectrogram(y=signal, sr=sr, n_mels=128)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    img = librosa.display.specshow(mel_db, sr=sr, x_axis='time', y_axis='mel', ax=axes[row_idx, 1])
    axes[row_idx, 1].set_title(f'Mel-спектрограмма: {row.coarse_group} / {row.category}')
    axes[row_idx, 1].set_xlabel('Время')
    axes[row_idx, 1].set_ylabel('Mel')

plt.tight_layout()
plt.show()

display(example_rows[['filename', 'target', 'category', 'coarse_group', 'fold']])

Примеры волновых форм и mel-спектрограмм показывают, почему частотно-временные представления являются сильным baseline для этой задачи: у многих классов заметно различаются временная огибающая и спектральная текстура.

## Прослушивание примера

In [ ]:
sample_row = esc.sample(1, random_state=RANDOM_STATE).iloc[0]
print(sample_row[['filename', 'target', 'category', 'coarse_group', 'fold']])
ipd.Audio(sample_row['audio_path'])

## Итоги

ESC-50 — аккуратный benchmark для классификации звуков окружающей среды: 2 000 клипов, 50 сбалансированных классов, 5 официальных folds и 40 клипов на класс. Главное следствие для моделирования: валидацию лучше строить по заданному fold-разбиению; случайное clip-level разбиение может завышать обобщающую способность, потому что клипы могут быть получены из одних и тех же исходных записей.